# Separating the Data Generating Process from Estimation


## Looking back at the Monopoly example

In the previous lecture we wrote a function to estimate the probability distribution over how many turns it takes to complete one lap in Monopoly. Here is that function again:

In [ ]:
from random import randint
from collections import defaultdict

def estimate_monopoly_probability(num_experiments: int = 1000) -> dict:
    counts = defaultdict(int)
    for experiment in range(num_experiments):
        square = 0
        num_throws = 0
        while square < 40:
            throw = sum([randint(1, 6) for i in range(2)])
            square += throw
            num_throws += 1
        counts[num_throws] += 1
    probability_distribution = {num_throws: count / num_experiments
                                 for num_throws, count in counts.items()}
    return probability_distribution

print(estimate_monopoly_probability())

This works. But look carefully at what it is actually doing. There are two quite different things happening inside the same function.


## Two things, not one

The function above mixes together:

1. **A simulation of one Monopoly turn** — rolling two dice, moving along the board, repeating until square 40 is passed, and returning how many throws it took. This is completely specific to Monopoly.

2. **A loop that runs that simulation many times and counts outcomes** — this is how we estimate a probability distribution from repeated sampling. It has nothing to do with Monopoly.

These are two conceptually separate ideas that happen to be written as one. Let us pull them apart.

## Part 1: The data generating process

The first part is a **data generating process** (DGP). A DGP is a model of the mechanism that produces one outcome — one sample from the process we are studying.


In [ ]:
def sample_from_monopoly_dgp() -> int:
    square = 0
    num_throws = 0
    while square < 40:
        throw = sum([randint(1, 6) for i in range(2)])
        square += throw
        num_throws += 1
    return num_throws

This function does exactly one thing: it plays out one lap of Monopoly and returns how many throws were needed. It contains all the domain knowledge — the board has 40 squares, you throw two dice, and so on.

Crucially, this is also an **exact** reproduction of the process. It does not approximate anything. The randomness in the function is the same randomness that exists in the real game. Each call produces one genuine sample from the Monopoly probability distribution.

## Part 2: The estimation

The second part is a **Monte Carlo estimator**. Given any DGP, it estimates the probability distribution over outcomes by running the DGP many times and counting how often each outcome appears.

In [ ]:
def estimate_probability_distribution(dgp, num_experiments: int = 1000) -> dict:
    counts = defaultdict(int)
    for experiment in range(num_experiments):
        outcome = dgp()
        counts[outcome] += 1
    probability_distribution = {outcome: count / num_experiments
                                 for outcome, count in counts.items()}
    return probability_distribution

Notice what this function does *not* contain: anything about Monopoly, dice, boards, or throws. It just calls `dgp()`, records what comes out, and aggregates. It would work equally well for a coin toss DGP, a temperature simulation, or any other process.

## The main point: conceptual clarity

The reason to make this separation is not primarily about writing cleaner code. It is about seeing clearly what kind of thing each part is.

**The DGP is a claim about the world.** When we write `sample_from_monopoly_dgp`, we are saying: *this is how the process actually works*. Two dice, 40 squares, advance until you pass. This is domain knowledge encoded in code. If our understanding of Monopoly were wrong, the DGP would be wrong, and everything downstream would be wrong too.

**The estimation is a generic procedure.** The Monte Carlo loop does not know or care about Monopoly. It only knows how to approximate a probability distribution from samples: run the process many times, count outcomes, divide by the number of runs. This procedure would work for any DGP. It is the same method regardless of the problem.

This distinction matters beyond Monopoly. In any simulation-based analysis, there will always be these two components:

- A **problem-specific** part that models the mechanism generating data
- A **generic** part that estimates probabilities from the samples that mechanism produces

Keeping them separate forces us to be precise about which part we are reasoning about when something goes wrong or when we want to change the model.

## A note on approximation

There is an asymmetry worth noting. The DGP is **exact** — each call genuinely samples from the distribution we are trying to study. The estimator, on the other hand, **approximates** — it gives us an empirical distribution that converges to the true distribution as `num_experiments` grows, but is never exactly equal to it.

This means that when results look wrong, we know where to look:

- If the model behaves unrealistically → the DGP is wrong
- If the estimates are noisy or unstable → `num_experiments` is too small

The separation makes these two sources of error clearly distinguishable.

## A secondary point: code design

As a side effect, the separation also gives us better code. `estimate_probability_distribution` can now be reused with any DGP we write. We saw this with coin tosses in the previous lecture — the estimation logic was always the same. We were just rewriting it each time without noticing.


In [ ]:
# The same estimator works for coins:
def sample_from_coin_dgp(p_head: float = 0.5) -> str:
    return 'H' if random.random() < p_head else 'T'

# We need a wrapper since our estimator expects a zero-argument callable:
estimate_probability_distribution(lambda: sample_from_coin_dgp(p_head=0.5))

But again — the reusability is a consequence of the conceptual separation, not the goal of it.
